# Any-Precision: CNN-8 op SVHN
## Proefrun van vijf volledige epochs

Dit notebook **leest uitsluitend opgeslagen resultaten**. Het traint niet, downloadt geen dataset en wijzigt geen checkpoints. De run gebruikt de eerste vijf epochs van het oorspronkelijke 100-epochrecept. De paperwaarden na 100 epochs zijn context, **geen slaagcriterium voor deze proef**.

Bron: *Any-Precision Deep Neural Networks*, p. 5 (recept), p. 6 / gedrukte p. 10768, tabel 1, CNN-8, kolommen **Ours**. De aangeleverde `train.py` bepaalt de checkpointselectie.

In [ ]:
from pathlib import Path
RUN_DIR = Path('C:\\Users\\renzo\\master-thesis-mx\\pilot_runs\\anyprecision_svhn_pilot5\\20261016_143302_354821_run')


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def read_json(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))

required = ["run_report.json", "epoch_metrics.csv", "selected_metrics.csv",
            "checkpoint_selection.json", "epoch_times.csv", "timing_estimate.json"]
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Geen complete resultaten in {RUN_DIR}: {missing}. Gebruik het notebook uit de resultatenmap.")
report = read_json("run_report.json")
metrics = pd.read_csv(RUN_DIR / "epoch_metrics.csv")
selected = pd.read_csv(RUN_DIR / "selected_metrics.csv")
times = pd.read_csv(RUN_DIR / "epoch_times.csv")
selection = read_json("checkpoint_selection.json")
estimate = read_json("timing_estimate.json")
if report.get("synthetic_fixture_only"):
    display(Markdown("**LET OP: synthetische softwaretest. Geen SVHN-onderzoeksresultaten of GPU-tijdmeting.**"))
display(Markdown(f"**Status:** {report['status']} &nbsp; | &nbsp; **Afgeronde epochs:** {report.get('completed_epoch', 0)}"))
display(pd.DataFrame([report.get("environment", {})]).T.rename(columns={0: "waarde"}))


## 1. Vastgelegd protocol en gegevens

Training: `train + extra`, batch 128, Adam met learning rate 0,001 en geen weight decay. Het LR-schema blijft 50/75/90 binnen 100 epochs. Tijdens deze proef is de learning rate dus niet vervroegd verlaagd. De vijf precisies gebruiken recursieve distillatie. Evaluatie vindt na elke epoch plaats op de volledige testset. `val` is de naam uit de broncode, niet een aparte validatieset.

In [ ]:
display(pd.DataFrame(list(report.get("config", {}).items()), columns=["instelling", "waarde"]))
display(pd.DataFrame(list(report.get("execution_settings", {}).items()), columns=["uitvoering", "waarde"]))
if "dataset_selection" in report:
    display(pd.DataFrame([{"split": name, **report["dataset_selection"][name]} for name in ["train", "test"]]))
else:
    display(Markdown("Geen productie-datasetmanifest in deze softwaretest."))
# Controleer de geregistreerde training, niet alleen een handmatig ingevulde titel.
assert sorted(metrics.bits_w_and_a.unique()) == [1, 2, 4, 8, 32]
assert metrics.objective_loss.notna().all()
assert (metrics.lr == 0.001).all(), "Onverwachte LR in de eerste vijf epochs."


## 2. Leercurves

Trainingsaccuracy is gemeten met dropout actief en verandert gedurende een epoch. Testaccuracy is gemeten na de epoch met het model in evaluatiemodus. Een lagere precisie mag een hogere score halen; er wordt geen rangorde afgedwongen.

In [ ]:
for phase, title in [("train", "Trainingsaccuracy"), ("test", "Testaccuracy")]:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for bit in [1, 2, 4, 8, 32]:
        rows = metrics[(metrics.phase == phase) & (metrics.bits_w_and_a == bit)].sort_values("epoch")
        ax.plot(rows.epoch, rows.top1_percent, marker="o", label=f"W{bit}/A{bit}")
    ax.set(title=title, xlabel="Epoch", ylabel="Top-1-accuracy (%)")
    ax.set_xticks(sorted(metrics.epoch.unique()))
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=3)
    fig.tight_layout()
    plt.show()


## 3. Beste checkpoint en laatste epoch

Het beste checkpoint is geselecteerd met de **hoogste W32/A32-testaccuracy**. De eerste epoch wint bij een gelijke score, zoals bij de strikte `>`-vergelijking in de code. Alle precisies in een rij horen bij hetzelfde checkpoint. Deze testscore is niet onafhankelijk van de checkpointselectie. Het laatste checkpoint wordt afzonderlijk behouden, ook wanneer beide rollen dezelfde epoch aanduiden.

In [ ]:
display(pd.DataFrame([{"rol": "beste volgens W32/A32-test", **selection["best"]},
                      {"rol": "laatste epoch", **selection["last"]}]))
comparison = selected.pivot(index=["role", "epoch"], columns="bits_w_and_a", values="top1_percent")
comparison.columns = [f"W{b}/A{b} (%)" for b in comparison.columns]
display(comparison.round(4))
paper = read_json("paper_targets.json")
paper_row = pd.DataFrame([paper["accuracy_percent_by_bits"]], index=["Paper: CNN-8 / Ours / 100 epochs"])
paper_row.columns = [f"W{b}/A{b} (%)" for b in paper_row.columns]
display(paper_row)
display(Markdown("**Let op:** vijf epochs en honderd epochs zijn geen gelijke trainingsbudgetten. Verschillen zijn hier geen bewijs voor of tegen paperreproductie."))


## 4. Tijdmeting en schatting voor 100 epochs

De gemeten epochtijd omvat training, evaluatie bij alle precisies, controles en checkpointopslag. Download, datasetinladen en de uiteindelijke rapportexport zijn afzonderlijke opstart-/afsluitkosten. De eerste epoch van ieder proces wordt niet gebruikt als warme epoch. De centrale extrapolatie is de tijd van epoch 1 plus 99 maal de mediaan van de overige gemeten warme epochs. De band op basis van de kleinste en grootste warme epochtijd is **geen statistisch betrouwbaarheidsinterval**.

In [ ]:
time_columns = ["epoch", "train_seconds", "test_seconds", "checkpoint_seconds", "other_seconds", "total_epoch_seconds"]
display(times[time_columns].round(3))
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, label in [("train_seconds", "Training"), ("test_seconds", "Test"), ("total_epoch_seconds", "Totaal")]:
    ax.plot(times.epoch, times[name] / 60, marker="o", label=label)
ax.set(xlabel="Epoch", ylabel="Minuten", title="Werkelijk gemeten duur per epoch")
ax.set_xticks(times.epoch)
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()
if estimate.get("available"):
    low, high = estimate["observed_range_projection_seconds"]
    display(pd.DataFrame([{
        "geschatte_100_epochs_uur": estimate["estimated_100_epoch_seconds"] / 3600,
        "projectie_min_uur": low / 3600, "projectie_max_uur": high / 3600,
        "gebruikte_warme_epochs": str(estimate["steady_epochs"]),
    }]))
else:
    display(Markdown(estimate.get("reason", "Nog geen tijdschatting.")))
if "data_preparation_seconds_including_download_and_checksums" in report:
    display(pd.DataFrame([{
        "download_en_controles_seconden": report["data_preparation_seconds_including_download_and_checksums"],
        "dataset_inladen_seconden": report["dataset_load_seconds"],
        "volledige_sessie_seconden": report["session_wall_seconds"],
    }]))


## 5. Geheugen, controles en beperkingen

De GPU-cijfers zijn PyTorch-geheugenmetingen, geen energie- of MX-hardwaremetingen. Met `cudnn.benchmark=True` volgen we de auteurscode, maar beloven we geen bitexact identieke CUDA-herhaling. De volledige model-, optimizer-, scheduler- en toevalstoestanden worden wel opgeslagen. Compatibiliteitswijzigingen staan in `compatibility.patch`; model, kwantisator en loss blijven uit het aangeleverde archief.

In [ ]:
memory = times[["epoch", "peak_gpu_allocated_bytes", "peak_gpu_reserved_bytes"]].copy()
for col in ["peak_gpu_allocated_bytes", "peak_gpu_reserved_bytes"]:
    memory[col.replace("_bytes", "_GiB")] = memory.pop(col) / 1024**3
display(memory)
checks = {key: report.get(key, "niet gerapporteerd") for key in
          ["source_unchanged", "dataset_files_unchanged", "optimizer_steps", "completed_epoch"]}
display(pd.DataFrame(list(checks.items()), columns=["controle", "waarde"]))
display(pd.DataFrame(report.get("warnings", [])))
for limitation in report.get("limitations", []):
    print(limitation)


## 6. Interpretatie na de proef

Noteer na controle van de echte uitvoer: of de technische controles slaagden; welke epochtijden representatief lijken; of de totale geschatte looptijd haalbaar is; en of er waarschuwingen of ontbrekende gegevens zijn. De 100-epochreferentie wordt daarna als een **nieuwe run vanaf initialisatie** gestart, niet automatisch uit deze proef voortgezet. Resultaten van deze proef worden niet overschreven.